# First causal GRU experiment

Use the same six sensor features, coverage rule, and participant split as the baseline.
The GRU receives at most ten completed epochs. It predicts the stage of the final epoch.
Short histories are retained after each sequence boundary. The model ignores padding.
The first run uses unweighted cross-entropy. This avoids adding class weighting alongside the architecture change.
Training stays under operator control. Report generation skips the tagged training cell.


In [ ]:
import json
import logging
import os
import shutil
import sys
from pathlib import Path

import pandas as pd
from IPython import get_ipython
from IPython.display import display

logging.basicConfig(
    level=logging.INFO,
    stream=sys.stdout,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
    force=True,
)
logger = logging.getLogger("gru_notebook")
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/gru.yaml").exists()),
    Path("/content/sleep-classifier"),
)
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "main"  # Select experiment/causal-gru before the PR merges
RUN_SETUP = False
RUN_PREPARATION = False
RUN_CHECK = True
RUN_TRAINING = False

## 1. Optional environment setup

Enable RUN_SETUP in a fresh Colab runtime. Select experiment/causal-gru while reviewing the PR.
Use main after merge. Choose a CPU runtime locally or CUDA in Colab when available.
Actual Colab/CUDA execution remains unverified. Local Mac execution currently uses CPU rather than MPS.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if not (ROOT / "configs/gru.yaml").exists():
        raise RuntimeError("Select the GRU experiment branch before the PR merges")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Installer download failed")
        !sh "{installer}" --no-modify-path
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Dependency installation failed")
    !git rev-parse HEAD

## 2. Optional data preparation

Enable RUN_PREPARATION for a fresh checkout. These commands prepare data without training a classifier.
The existing split is immutable. Final test observations receive integrity checks but remain outside model preparation.


In [ ]:
if RUN_PREPARATION:
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -m sleep_classifier.download_cohort \
        --archive data/sleep-accel-1.0.0.zip --destination data/raw/sleep-accel/1.0.0
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Cohort download failed")
    !uv run --locked --group gru --group notebook python -m sleep_classifier.cohort \
        inventory --config configs/cohort.yaml
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Inventory failed")
    !uv run --locked --group gru --group notebook python -m sleep_classifier.cohort \
        split --config configs/cohort.yaml
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Split verification failed")
    !uv run --locked --group gru --group notebook python -m sleep_classifier.prepare \
        --config configs/prepare.yaml
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Feature preparation failed")

## 3. Preflight without training

This command loads development features, fits the training-only scaler, and runs one no-gradient forward pass.
It creates no trained checkpoint or validation metrics. Read configs/gru.yaml before the actual training step.


In [ ]:
if RUN_CHECK:
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -m sleep_classifier.gru_training \
        --config configs/gru.yaml --check
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("GRU preflight failed")

## 4. Train when ready

Enable RUN_TRAINING only when you choose to fit the GRU.
The first configuration has 32 hidden units, at most 20 training epochs, and patience of five epochs.
Each training epoch visits all retained training targets. Validation macro-F1 selects the best checkpoint.
The checkpoint restores that best epoch before final validation export.
Logs report the epoch number, training loss, validation loss, accuracy, and macro-F1.
Changing the output directory preserves results from previous runs.


In [ ]:
if RUN_TRAINING:
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -u -m sleep_classifier.gru_training \
        --config configs/gru.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("GRU training failed")

## 5. Inspect saved results

Metrics apply to retained validation epochs. Compare them with the unweighted logistic reference first.
The random model used by preflight has no reported classification performance.
A GRU gain would combine architecture and history effects. A later one-epoch-context GRU can test the value of history separately.


In [ ]:
metrics_path = ROOT / "artifacts/gru-context-10/metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    logger.info(
        "Best epoch: %s; accuracy: %.4f; macro-F1: %.4f",
        metrics["best_epoch"],
        metrics["accuracy"],
        metrics["macro_f1"],
    )
    display(pd.DataFrame(metrics["history"]))
    display(pd.DataFrame(metrics["classes"]).T)
    display(
        pd.DataFrame(
            metrics["confusion_matrix"],
            index=["Wake", "Light", "Deep", "REM"],
            columns=["Wake", "Light", "Deep", "REM"],
        )
    )
else:
    logger.info("No GRU training results exist. Only preflight has been run.")

In [ ]:
if metrics_path.exists():
    import matplotlib.pyplot as plt
    import numpy as np

    history = pd.DataFrame(metrics["history"])
    figure, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    axes[0].plot(history.epoch, history.train_loss, label="Training")
    axes[0].plot(history.epoch, history.validation_loss, label="Validation")
    axes[0].set(xlabel="Epoch", ylabel="Cross-entropy loss")
    axes[0].legend()
    axes[1].plot(history.epoch, history.validation_macro_f1, label="Macro-F1")
    axes[1].plot(history.epoch, history.validation_accuracy, label="Accuracy")
    axes[1].axvline(metrics["best_epoch"], color="gray", linestyle="--", label="Selected epoch")
    axes[1].set(xlabel="Epoch", ylabel="Validation score", ylim=(0, 1))
    axes[1].legend()
    counts = np.asarray(metrics["confusion_matrix"])
    fractions = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
    axes[2].imshow(fractions, vmin=0, vmax=1, cmap="Blues")
    stages = ["Wake", "Light", "Deep", "REM"]
    axes[2].set_xticks(range(4), stages, rotation=25)
    axes[2].set_yticks(range(4), stages)
    axes[2].set(xlabel="Predicted stage", ylabel="True stage")
    for row in range(4):
        for column in range(4):
            axes[2].text(
                column,
                row,
                f"{fractions[row, column]:.0%}",
                ha="center",
                va="center",
                color="white" if fractions[row, column] > 0.5 else "black",
            )
    figure.tight_layout()
    figure_dir = metrics_path.parent / "figures"
    figure_dir.mkdir(exist_ok=True)
    figure.savefig(figure_dir / "training-and-validation.png", dpi=150)
    plt.show()

## Export HTML

```bash
uv run --locked --group gru --group notebook python scripts/execute_notebook.py --notebook notebooks/07_causal_gru.ipynb
```

The report executes preflight and reads existing results. Setup, preparation, and training cells are skipped.
Source notebooks remain output-free. Executed reports are HTML only.


In [ ]:
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/07_causal_gru.ipynb